# Methodology: overview

This part is a self-contained statistical reference for the methods `mboostrs`
implements. It assumes familiarity with regression and basic optimization but
develops the boosting machinery from first principles. Code is deferred to
Part I; here we work with equations.

```{admonition} Where to start
:class: tip

If you are new to model-based boosting, read the chapters **in order**:

1. This overview ([](overview.ipynb)) for the big picture and notation.
2. [](prerequisites.ipynb) for the mathematical foundations (linear algebra,
   optimization, GLMs, GAMs) that the rest assumes.
3. [](from_glm_to_boosting.ipynb) for the conceptual journey motivating every
   design choice.
4. [](functional_gradient_boosting.ipynb) for the core algorithm.
5. [](theory_properties.ipynb) for the convergence and consistency theory.

If you already know boosting and want the GAMLSS material, jump to
[](gamlss_framework.ipynb) (the framework) then [](gamlss.ipynb) (boosting GAMLSS).
```

## Notation

We observe data $\{(y_i, x_i)\}_{i=1}^n$ with weights $w_i \ge 0$. A model is an
additive **predictor** $f(\cdot)$ built from $J$ candidate **base learners**
$g_1, \dots, g_J$. Each base learner $g_j$ maps the data to a fitted vector and
is itself estimated by penalized least squares. We write:

- $\rho(y, f)$ — the **loss** (the family); the empirical risk is
  $\mathcal R(f) = \sum_i w_i\, \rho(y_i, f_i)$.
- $u = -\partial \mathcal R/\partial f$ — the **negative gradient** of the risk
  with respect to the predictor evaluated at the current fit (the "pseudo-
  residuals").
- $\nu \in (0,1]$ — the **step length** (learning rate).
- $m_{\text{stop}}$ — the number of boosting iterations.
- $H_j$ — the **hat matrix** of base learner $j$ (the linear map from a response
  to its fitted values), used for degrees-of-freedom accounting.

## Family, learners and regularization

A boosting model is determined by three independent choices, each treated in its
own chapter:

1. **The family** ([](families.ipynb)) — what is being estimated. The loss encodes
   the target functional (mean, quantile, hazard, a distribution parameter) and
   supplies the negative gradient $u$ and the offset $\hat f^{[0]}$.

2. **The base learners** ([](base_learners.ipynb), [](penalties_df.ipynb)) — the space
   of candidate effects. Linear terms, P-splines, monotone splines, spatial,
   random, and tree learners, each a penalized least-squares fit with degrees of
   freedom controlled by `df`.

3. **The fitting algorithm** ([](functional_gradient_boosting.ipynb)) and its
   **regularization** ([](model_selection.ipynb)) — component-wise gradient descent
   in function space, stopped early at an $m_{\text{stop}}$ chosen by
   cross-validated risk.

Distributional regression ([](gamlss_framework.ipynb) for the framework,
[](gamlss.ipynb) for the boosting version) layers a fourth choice — *which
distribution parameter* a learner updates — on top, and multinomial models
([](multinomial.ipynb)) use a structured (Kronecker) design across classes. The
statistical theory underpinning all of the above — convergence, consistency,
and oracle inequalities — is collected in [](theory_properties.ipynb).

## Properties of component-wise boosting

Component-wise boosting selects effects during fitting and limits their cumulative
complexity through early stopping:

- **Learner selection.** Each iteration adds one candidate learner. Learners
  never selected before $m_{\text{stop}}$ contribute zero, so early stopping
  can produce a sparse model even when $J \gg n$.

- **Early stopping.** At $m_{\text{stop}}=0$ the model contains only the offset.
  Increasing $m_{\text{stop}}$ permits more updates and reduces shrinkage along
  the fitted path. Cross-validated risk selects the stopping iteration; an
  unpenalized limiting fit requires the conditions described in
  [](theory_properties.ipynb).

- **Loss families.** Changing the loss $\rho$ adapts
  the same algorithm to classification, robust regression, quantiles, survival,
  counts, and every parameter of a distribution — with no change to the base
  learners or the engine.